## Ingest Constructors Data (JSON to Delta Lake)

This notebook reads the **constructors.json** file (containing Formula 1 team/constructor details) from the landing volume and loads it into a Bronze Delta table.

**What's different here?** Unlike circuits and races (which are CSV files), constructors data comes as a **JSON file**. The read process is slightly different but the overall pattern remains the same.

**Steps:**
1. **Define schema** and **read** the JSON file
2. **Enrich** with metadata columns (ingestion timestamp + source file)
3. **Write** to the Bronze Delta table `formula1.bronze.constructors`
4. **Verify** the data was written correctly

#### Loading Configuration
We run two shared notebooks to load reusable variables and helper functions:
- **`01.environment-config`** - catalog name, schema names, file paths
- **`02.bronze_helpers`** - the `add_ingestion_metadata()` function

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze_helpers

#### Step 4: Verify the Data
As a final check, we read back the table we just wrote to confirm everything landed correctly. If you see rows with all columns populated, the ingestion worked!

In [0]:
source_file =f'{loding_folder_path}/constructors.json' # to replace the load in read api
table_name = f'{catalog_name}.{bronze_schema}.constructors'

#### Define the Schema (DDL Format)
For JSON files, we can define the schema as a simple **DDL string** (like SQL column definitions) instead of using StructType. This is shorter and easier to read.

**What's happening:**
- We list each column name followed by its data type
- `STRING` means text, `INT` means whole number
- Spark will only read these 4 columns from the JSON, ignoring any extras

In [0]:
from pyspark.sql.types import *
constructor_schema = 'constructorId STRING, name STRING, nationality STRING, url STRING'

#### Step 1: Read the JSON File
We use Spark's DataFrame Reader to load `constructors.json` from the landing volume.

**What's happening in the code below:**
- `format('json')` - tells Spark the file is JSON (not CSV)
- `.schema(constructor_schema)` - applies our DDL schema defined above
- `.load(source_file)` - reads from the path stored in our variable

**Note:** JSON files don't need a `header` option like CSV - each record already has field names built in.

In [0]:
constructors_df = (
    spark.read
    .format('json')
    .option('Headers',True)
    # .option('inferSchema', True)
    .schema(constructor_schema)
    .load(source_file)
)
display(constructors_df)


#### Step 2: Add Metadata Columns
We call the shared helper function `add_ingestion_metadata()` to add two tracking columns:
- **`ingestion_timestamp`** - when this data was loaded
- **`source_file`** - which file the row came from

This function lives in the **`02.bronze_helpers`** notebook that we loaded at the top.

In [0]:
constructors_final_df = add_ingestion_metadata(constructors_df)


#### Step 3: Write to Bronze Delta Table
We save the enriched DataFrame as a Delta table (`formula1.bronze.constructors`).

**What's happening:**
- `mode('overwrite')` - replaces the entire table each run (clean reload)
- `format('delta')` - saves in Delta format (supports versioning and fast queries)
- `saveAsTable(table_name)` - registers it in Unity Catalog so anyone can query it with SQL

In [0]:
(
    constructors_final_df
    .write
    .mode('overwrite')
    .format('delta')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))